# FeynmanEngine — Getting Started

This notebook walks through a complete workflow with FeynmanEngine: install, import, generate Feynman diagrams, compute scattering amplitudes, integrate cross-sections, make differential distributions, and (optionally) launch the browser UI.

Every numerical result returned by the engine includes a **trust badge** — `validated`, `approximate`, `rough`, or `blocked` — so you always know how much to trust the answer.

## What you'll do

1. Install the package + native dependencies
2. Inspect the engine and check what's installed (`feynman doctor`)
3. Generate Feynman diagrams for `e⁺e⁻ → μ⁺μ⁻`
4. Compute the symbolic spin-averaged $|\overline{\mathcal{M}}|^2$
5. Integrate the cross-section at LEP-2 energies (LO + NLO)
6. Make a differential distribution $d\sigma/d\cos\theta$
7. Compute a hadronic cross-section: $pp \to t\bar{t}$ at the LHC
8. Try a process the engine refuses (the trust system in action)
9. Launch the browser UI

## 1. Install

FeynmanEngine is on PyPI. The package itself is pure Python; the native HEP tools (QGRAF, FORM, LoopTools, LHAPDF) are compiled from source via the `feynman setup` command after install.

```bash
pip install feynman-engine
feynman setup --with-lhapdf       # builds QGRAF + FORM + LoopTools + LHAPDF (~5-10 min, one-time)
```

If you only want a quick start without LHAPDF, drop `--with-lhapdf`; the engine falls back to its built-in PDF (factor-of-2-3 accuracy for hadronic cross-sections).

Or with Docker (everything bundled):

```bash
docker run -p 8000:8000 ecavan/feynman-api:latest
# Open http://localhost:8000
```

## 2. Check what's installed

In [ ]:
from feynman_engine.diagnostics import collect_diagnostics, summarize_doctor_report

diagnostics = collect_diagnostics()
print(summarize_doctor_report(diagnostics))

All five components should report `ok` after `feynman setup --with-lhapdf`. If LHAPDF is missing the engine still works — hadronic cross-sections just use the built-in PDF instead of CT18LO.

## 3. Generate Feynman diagrams

Pick a process, theory, and loop order. The engine routes through QGRAF for diagram enumeration, then FORM/SymPy for symbolic algebra.

In [ ]:
from feynman_engine.engine import FeynmanEngine

engine = FeynmanEngine()
result = engine.generate(
    process="e+ e- -> mu+ mu-",
    theory="QED",
    loops=0,
    output_format="svg",
)

print(f"Generated {result.summary['total_diagrams']} diagram(s)")
for d in result.diagrams:
    print(f"  Diagram {d.id}: {d.topology}")

In [ ]:
# Render the first diagram inline
from IPython.display import SVG, display

first_id = result.diagrams[0].id
svg_bytes = result.images.get(first_id)
if svg_bytes:
    display(SVG(svg_bytes))

## 4. Get the symbolic |M|²

In [ ]:
from feynman_engine.physics.amplitude import get_amplitude

amp = get_amplitude("e+ e- -> mu+ mu-", "QED")
print(f"Backend: {amp.backend}")
print(f"|M̄|² = {amp.msq}")
print(f"Description: {amp.description}")

## 5. Integrated cross-section: LO and NLO

For $e^+e^- \to \mu^+\mu^-$ in QED at $\sqrt{s} = 91$ GeV, the analytic formula is $\sigma = 4\pi\alpha^2/(3s) \approx 10.5$ pb.

In [ ]:
from feynman_engine.amplitudes.cross_section import total_cross_section
from feynman_engine.physics.trust import classify

for order in ("LO", "NLO"):
    r = total_cross_section("e+ e- -> mu+ mu-", "QED", sqrt_s=91.0)
    trust = classify("e+ e- -> mu+ mu-", "QED", order)
    print(f"σ_{order} = {r['sigma_pb']:.3f} pb  [trust: {trust.trust_level.value}]")
    print(f"  reference: {trust.reference}")

The `trust_level` field tells us this matches the textbook formula. For NLO, the engine returns the exact analytic K-factor $K = 1 + 3\alpha/(4\pi) \approx 1.0017$.

## 6. Differential distribution $d\sigma/d\cos\theta$

In [ ]:
import numpy as np
from feynman_engine.amplitudes.differential import differential_distribution

edges = np.linspace(-1, 1, 21)
hist = differential_distribution(
    "e+ e- -> mu+ mu-", "QED", sqrt_s=10.0,
    observable="cos_theta", bin_edges=edges,
)
print(f"σ_total = {hist['sigma_total_pb']:.1f} pb")

In [ ]:
import matplotlib.pyplot as plt

centers = np.array(hist["bin_centers"])
ds_dc   = np.array(hist["dsigma_dX_pb"])

plt.figure(figsize=(7, 4))
plt.bar(centers, ds_dc, width=0.09, color="steelblue", edgecolor="navy")
plt.xlabel("cos θ")
plt.ylabel("dσ/d(cos θ) [pb]")
plt.title(r"$e^+ e^- \to \mu^+ \mu^-$ at $\sqrt{s}=10$ GeV — characteristic $1+\cos^2\theta$ shape")
plt.tight_layout()
plt.show()

## 7. Hadronic cross-section: $pp \to t\bar{t}$ at the LHC

Switch to a hadron-collider process. The engine convolves partonic $\hat\sigma$ with parton distribution functions; with LHAPDF + CT18LO this matches the LHC LO reference within ~13%.

In [ ]:
from feynman_engine.amplitudes.hadronic import hadronic_cross_section

for order in ("LO", "NLO"):
    r = hadronic_cross_section("p p -> t t~", sqrt_s=13000.0, theory="QCD", order=order)
    K = r.get("k_factor", 1.0)
    print(f"σ_{order}(pp → tt̄, 13 TeV) = {r['sigma_pb']:.0f} pb  (K = {K:.2f})")
print(f"\nPDF: {r['pdf']}")
print("Reference (LHC HWG YR4): σ_LO ≈ 700 pb, σ_NLO ≈ 830 pb")

## 8. Trust system in action — refused process

The engine will **refuse** to return a number for processes it knows are wrong. For `pp → W⁺W⁻` we only have the t-channel quark contribution (missing s-channel γ + Z + interference); the engine flags this as `BLOCKED`.

In [ ]:
from feynman_engine.physics.trust import classify

trust = classify("p p -> W+ W-", "EW", "LO")
print(f"Trust level: {trust.trust_level.value}")
if trust.block_reason:
    print(f"Reason: {trust.block_reason}")
    print(f"Workaround: {trust.workaround}")

In [ ]:
# Through the API this returns HTTP 422 — never a wrong number
from fastapi.testclient import TestClient
from feynman_engine.api.app import app

client = TestClient(app)
r = client.get("/api/amplitude/cross-section", params={
    "process": "p p -> W+ W-", "theory": "EW", "sqrt_s": 13000,
})
print(f"HTTP {r.status_code}")
import json
print(json.dumps(r.json(), indent=2))

## 9. Launch the browser UI

FeynmanEngine ships with an interactive frontend (FastAPI + tabbed UI for diagrams and differential distributions). Start the server in a background cell:

In [ ]:
import subprocess
import sys
import time

# Launch in background
server = subprocess.Popen(
    [sys.executable, "-m", "feynman_engine", "serve",
     "--host", "127.0.0.1", "--port", "8765"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)  # give it a moment to bind

from IPython.display import IFrame, Markdown, display
display(Markdown("### → [Open in browser](http://localhost:8765)"))
display(Markdown("Or render inline below:"))
display(IFrame("http://localhost:8765", width="100%", height=600))

In [ ]:
# When done, stop the server:
server.terminate()
server.wait()
print("Server stopped.")

## What's next

- **Browse all processes**: `GET /api/amplitude/processes` returns the full list of curated entries (or call `feynman_engine.physics.amplitude.list_user_amplitudes()` from Python).
- **Register your own amplitude**: `feynman_engine.physics.amplitude.register_curated_amplitude(process, theory, msq=...)` to plug in a custom |M̄|².
- **Different PDFs**: pass `pdf_name="NNPDF40_lo_as_01180"` (after `feynman install-pdf-set NNPDF40_lo_as_01180`) to use a different LHAPDF set.
- **Swagger docs**: `http://localhost:8765/docs` for the full API surface.

Citation guidance and the list of bundled HEP tools are in the README.